In [ ]:
import os
import warnings

# from google.colab import drive
# drive.mount('/content/drive')
# base_path = "/content/drive/MyDrive/Colab Notebooks/Quant"
# os.chdir(base_path)
import sys
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


import tensorflow as tf
import keras

from config import config

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
warnings.filterwarnings("ignore", category=UserWarning, module="keras")

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from src.utils.sentiment_analysis_methods import (
    encode, get_vectorize_layer, get_masked_input_and_labels, augment_text, save_model_weights,
    create_masked_language_bert_model, MaskedTextGenerator, create_classifier_bert_model
)

# 1. Load Dataset

## 1.1 Load MLM Dataset

In [ ]:
news_text_raw = pd.read_csv(os.path.join("data", "abcnews-date-text.csv"))["headline_text"]
vectorize_layer = get_vectorize_layer(
    news_text_raw.tolist(),
    config.VOCAB_SIZE,
    config.MAX_LEN,
    special_tokens=["[mask]"],
)

mask_token_id = vectorize_layer(["[mask]"]).numpy()[0][0]
x_all_encoded = encode(news_text_raw, vectorize_layer=vectorize_layer)

x_masked_train, y_masked_labels, sample_weights = get_masked_input_and_labels(x_all_encoded, mask_token_id=mask_token_id)

mlm_ds = tf.data.Dataset.from_tensor_slices(
    (x_masked_train, y_masked_labels, sample_weights)
)

mlm_ds = mlm_ds.shuffle(1000).batch(config.BATCH_SIZE)
mlm_ds_small = mlm_ds.shard(num_shards=256, index=0)





## 1.2 Load Sentiment Dataset

In [ ]:
sentiment_raw = pd.read_csv(os.path.join("data", "sentiment.csv"), encoding='latin1', header=None)
sentiment_raw.columns = ["Output", "Input"]

df_neg = sentiment_raw[sentiment_raw["Output"] == "negative"]
df_neu = sentiment_raw[sentiment_raw["Output"] == "neutral"]
df_pos = sentiment_raw[sentiment_raw["Output"] == "positive"]

aug = naw.SynonymAug(aug_src='wordnet')
target_n = 2000 
df_neg_final = augment_text(df_neg, target_n, aug=aug)
df_neu_final = augment_text(df_neu, target_n, aug=aug)
df_pos_final = augment_text(df_pos, target_n, aug=aug)

final_df = pd.concat([df_neg_final, df_neu_final, df_pos_final]).sample(frac=1, random_state=42).reset_index(drop=True)

X_encoded = encode(final_df['Input']) 

le = LabelEncoder()
y_encoded = le.fit_transform(final_df['Output'])

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.1, random_state=42
)

train_classifier_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(len(X_train)).batch(32))
test_classifier_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(32)

# 2. Train

## 2.1 Pretrain w/ MLM

In [ ]:
loss_fn = keras.losses.SparseCategoricalCrossentropy(reduction=None)
loss_tracker = keras.metrics.Mean(name="loss")
id2token = dict(enumerate(vectorize_layer.get_vocabulary()))
token2id = {y: x for x, y in id2token.items()}
sample_tokens = vectorize_layer(["Google wont [mask] replacing our news headlines with terrible AI"])

bert_masked_model = create_masked_language_bert_model()

generator_callback = MaskedTextGenerator(sample_tokens.numpy())

earlyStopping_callback = keras.EarlyStopping(
    monitor="loss",
    min_delta=0.005,
    patience=5,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=True,
)

checkpoint_callback = keras.ModelCheckpoint(
    os.path.join("models", "model_4L_weights_cp_best.weights.h5"), 
    save_best_only=True,
    monitor="loss",
    save_weights_only=True,
    mode="min",
    verbose=1
)

bert_masked_model.fit(mlm_ds, epochs=50, callbacks=[generator_callback, checkpoint_callback, earlyStopping_callback])
save_model_weights(bert_masked_model, "bert_masked_model.weights.h5", os.path.join("model", "weights"))

## 2.2 Fine-tune

In [ ]:
pretrained_bert_model = keras.Model(
    bert_masked_model.input, bert_masked_model.get_layer("encoder_4_ffn_layernormalization").output
)

pretrained_bert_model.trainable = False

classifer_model = create_classifier_bert_model(pretrained_bert_model)

classifer_model.fit(
    train_classifier_ds,
    epochs=5,
    validation_data=test_classifier_ds,
)

In [ ]:
pretrained_bert_model.trainable = True

optimizer = keras.optimizers.Adam()

classifer_model.compile(
    optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

classifer_model.fit(
    train_classifier_ds,
    epochs=5,
    validation_data=test_classifier_ds,
)

save_model_weights(classifer_model, "classifier_model.weights.h5", os.path.join("model", "weights"))